# Course 19: Responsible AI — Interpretability & Transparency

**GCP Professional Machine Learning Engineer**  
**Exam Section 6: Responsible AI**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajvermatx/careeralign.com/blob/main/gcp-mle/notebooks/19-responsible-ai-interpretability.ipynb)

This notebook covers:
1. Train a model and extract feature importance
2. SHAP: summary plot and force plot
3. LIME: local explanations
4. Partial Dependence Plots
5. Permutation feature importance
6. Vertex AI Explainability configuration (SDK)
7. Attention visualization concept
8. Model Card template generator

---
## Setup & Installation

In [ ]:
!pip install -q shap lime matplotlib scikit-learn pandas numpy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

print("All imports successful!")

---
## 1. Train a Model & Extract Feature Importance

We use the Breast Cancer dataset (built into sklearn) — 30 features, binary classification.

In [ ]:
# Load dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Features: {list(X.columns[:5])}... ({len(X.columns)} total)")

In [ ]:
# --- Linear Model: Coefficient-based Feature Importance ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_model = LogisticRegression(penalty='l1', solver='liblinear', C=1.0, random_state=42)
lr_model.fit(X_train_scaled, y_train)

print(f"Logistic Regression Accuracy: {lr_model.score(X_test_scaled, y_test):.4f}")
print("\nTop 10 features by absolute coefficient:")
coef_df = pd.DataFrame({
    'feature': X.columns,
    'coefficient': lr_model.coef_[0]
}).assign(abs_coef=lambda df: df['coefficient'].abs())
coef_df = coef_df.sort_values('abs_coef', ascending=False)
print(coef_df[['feature', 'coefficient']].head(10).to_string(index=False))

In [ ]:
# --- Gradient Boosting: Impurity-based Feature Importance ---
gb_model = GradientBoostingClassifier(
    n_estimators=100, max_depth=3, random_state=42
)
gb_model.fit(X_train, y_train)

print(f"Gradient Boosting Accuracy: {gb_model.score(X_test, y_test):.4f}")

# Plot feature importance
imp_df = pd.DataFrame({
    'feature': X.columns,
    'importance': gb_model.feature_importances_
}).sort_values('importance', ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(imp_df['feature'], imp_df['importance'], color='#3b82f6')
ax.set_xlabel('Impurity-Based Feature Importance')
ax.set_title('Top 15 Features — Gradient Boosting')
plt.tight_layout()
plt.show()

---
## 2. SHAP: Summary Plot & Force Plot

SHAP (SHapley Additive exPlanations) assigns each feature a Shapley value representing its marginal contribution. Tree SHAP is exact and fast for tree-based models.

In [ ]:
import shap

# Use Tree SHAP (fast, exact for tree-based models)
explainer = shap.TreeExplainer(gb_model)
shap_values = explainer.shap_values(X_test)

print(f"SHAP values shape: {shap_values.shape}")
print(f"Each test sample gets {shap_values.shape[1]} SHAP values (one per feature)")

In [ ]:
# --- SHAP Summary Plot (Global) ---
# Shows feature importance AND the direction of impact
shap.summary_plot(shap_values, X_test, plot_type="dot", show=True)

In [ ]:
# --- SHAP Bar Plot (Global Feature Importance) ---
shap.summary_plot(shap_values, X_test, plot_type="bar", show=True)

In [ ]:
# --- SHAP Force Plot (Local — Single Prediction) ---
# Explain the first test sample
sample_idx = 0
print(f"Sample {sample_idx}: True label = {y_test.iloc[sample_idx] if hasattr(y_test, 'iloc') else y_test[sample_idx]}")
print(f"Model prediction = {gb_model.predict(X_test.iloc[[sample_idx]])[0]}")

shap.initjs()
shap.force_plot(
    explainer.expected_value,
    shap_values[sample_idx],
    X_test.iloc[sample_idx],
    matplotlib=True
)
plt.show()

In [ ]:
# --- SHAP Waterfall Plot (Local — Step-by-step breakdown) ---
shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[sample_idx],
        base_values=explainer.expected_value,
        data=X_test.iloc[sample_idx].values,
        feature_names=X_test.columns.tolist()
    ),
    max_display=15
)

---
## 3. LIME: Local Explanations

LIME fits a simple linear model in the neighborhood of a single prediction to explain it locally.

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

lime_explainer = LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=X_train.columns.tolist(),
    class_names=['malignant', 'benign'],
    mode='classification'
)

# Explain the same sample as SHAP
lime_exp = lime_explainer.explain_instance(
    data_row=X_test.iloc[sample_idx].values,
    predict_fn=gb_model.predict_proba,
    num_features=10
)

print("LIME Explanation for sample", sample_idx)
print("=" * 50)
for feature, weight in lime_exp.as_list():
    direction = "+" if weight > 0 else "-"
    print(f"  {direction} {feature}: {weight:.4f}")

In [ ]:
# LIME visual explanation
fig = lime_exp.as_pyplot_figure()
fig.set_size_inches(10, 6)
plt.title(f'LIME Explanation — Sample {sample_idx}')
plt.tight_layout()
plt.show()

In [ ]:
# Compare SHAP vs LIME top features for the same prediction
shap_top = pd.DataFrame({
    'feature': X_test.columns,
    'shap_value': np.abs(shap_values[sample_idx])
}).sort_values('shap_value', ascending=False).head(10)['feature'].tolist()

lime_top = [feat.split(' ')[0] for feat, _ in lime_exp.as_list()[:10]]

print("Top 10 features comparison:")
print(f"{'SHAP':<30} {'LIME':<30}")
print("=" * 60)
for s, l in zip(shap_top, lime_top):
    match = "  ✓" if s in lime_top else ""
    print(f"{s:<30} {l:<30}{match}")

---
## 4. Partial Dependence Plots (PDP)

PDPs show the marginal effect of a feature on the predicted outcome, averaged over all other features.

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

# Get top 4 features by importance
top_features = pd.DataFrame({
    'feature': X.columns,
    'importance': gb_model.feature_importances_
}).sort_values('importance', ascending=False).head(4)['feature'].tolist()

print(f"Plotting PDP for top features: {top_features}")

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for idx, feature in enumerate(top_features):
    ax = axes[idx // 2][idx % 2]
    PartialDependenceDisplay.from_estimator(
        gb_model, X_train, [feature],
        ax=ax, kind='both'  # 'both' = PDP line + ICE lines
    )
    ax.set_title(f'PDP + ICE: {feature}')

plt.suptitle('Partial Dependence Plots with ICE', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 2D PDP — Feature Interaction
fig, ax = plt.subplots(figsize=(8, 6))
PartialDependenceDisplay.from_estimator(
    gb_model, X_train,
    [(top_features[0], top_features[1])],  # 2D interaction
    ax=ax, kind='average'
)
ax.set_title(f'2D PDP: {top_features[0]} vs {top_features[1]}')
plt.tight_layout()
plt.show()

---
## 5. Permutation Feature Importance

Measures how much model performance degrades when a feature is shuffled. Unbiased w.r.t. cardinality.

In [ ]:
from sklearn.inspection import permutation_importance

# Compute on TEST set (not training set)
perm_result = permutation_importance(
    gb_model, X_test, y_test,
    n_repeats=30, random_state=42, scoring='accuracy'
)

perm_df = pd.DataFrame({
    'feature': X.columns,
    'importance_mean': perm_result.importances_mean,
    'importance_std': perm_result.importances_std
}).sort_values('importance_mean', ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(
    perm_df['feature'],
    perm_df['importance_mean'],
    xerr=perm_df['importance_std'],
    color='#22c55e', capsize=3
)
ax.set_xlabel('Mean Accuracy Decrease')
ax.set_title('Permutation Feature Importance (Test Set)')
plt.tight_layout()
plt.show()

In [ ]:
# Compare all three importance methods
comparison = pd.DataFrame({
    'impurity': gb_model.feature_importances_,
    'permutation': perm_result.importances_mean,
    'shap_mean': np.abs(shap_values).mean(axis=0)
}, index=X.columns)

# Normalize to [0, 1]
for col in comparison.columns:
    comparison[col] = comparison[col] / comparison[col].max()

top_10 = comparison.sort_values('shap_mean', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(top_10))
width = 0.25
ax.bar(x - width, top_10['impurity'], width, label='Impurity', color='#3b82f6')
ax.bar(x, top_10['permutation'], width, label='Permutation', color='#22c55e')
ax.bar(x + width, top_10['shap_mean'], width, label='SHAP', color='#a855f7')
ax.set_xticks(x)
ax.set_xticklabels(top_10.index, rotation=45, ha='right')
ax.set_ylabel('Normalized Importance')
ax.set_title('Feature Importance: Impurity vs Permutation vs SHAP')
ax.legend()
plt.tight_layout()
plt.show()

---
## 6. Vertex AI Explainability Configuration

Shows how to configure Vertex AI Explainability using the Python SDK.  
**Note**: API calls are commented out to avoid costs. Uncomment to run on GCP.

In [ ]:
# ============================================================
# Vertex AI Explainability — SDK Configuration
# ============================================================
# Uncomment and run on GCP with appropriate credentials.

# from google.cloud import aiplatform
#
# aiplatform.init(project="your-project", location="us-central1")
#
# # --- Option 1: Sampled Shapley (works with any model) ---
# explanation_params_shapley = aiplatform.explain.ExplanationParameters(
#     sampled_shapley_attribution=
#         aiplatform.explain.SampledShapleyAttribution(path_count=25)
# )
#
# # --- Option 2: Integrated Gradients (TensorFlow models) ---
# explanation_params_ig = aiplatform.explain.ExplanationParameters(
#     integrated_gradients_attribution=
#         aiplatform.explain.IntegratedGradientsAttribution(step_count=50)
# )
#
# # --- Option 3: XRAI (image models) ---
# explanation_params_xrai = aiplatform.explain.ExplanationParameters(
#     xrai_attribution=aiplatform.explain.XraiAttribution(step_count=50)
# )
#
# # Define explanation metadata (tabular example)
# explanation_metadata = aiplatform.explain.ExplanationMetadata(
#     inputs={
#         "features": aiplatform.explain.ExplanationMetadata.InputMetadata(
#             input_tensor_name="input_layer",
#             encoding="BAG_OF_FEATURES",
#             modality="numeric",
#             index_feature_mapping=list(X.columns),
#         )
#     },
#     outputs={
#         "prediction": aiplatform.explain.ExplanationMetadata.OutputMetadata(
#             output_tensor_name="output_layer"
#         )
#     },
# )
#
# # Upload model with explainability
# model = aiplatform.Model.upload(
#     display_name="breast-cancer-gb-model",
#     artifact_uri="gs://your-bucket/model/",
#     serving_container_image_uri=(
#         "us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-0:latest"
#     ),
#     explanation_parameters=explanation_params_shapley,
#     explanation_metadata=explanation_metadata,
# )
#
# # Deploy and get explanations
# endpoint = model.deploy(machine_type="n1-standard-4")
# prediction = endpoint.explain(instances=[X_test.iloc[0].tolist()])
# print("Feature attributions:", prediction.explanations[0].attributions)

print("Vertex AI Explainability code ready — uncomment to run on GCP.")
print("\nThree methods available:")
print("  1. Sampled Shapley  — any model type")
print("  2. Integrated Gradients — TensorFlow models")
print("  3. XRAI — image classification models")

---
## 7. Attention Visualization Concept

A simplified demonstration of how attention weights can be visualized in Transformer models.

In [ ]:
# Simulate attention weights for a simple sentence
# In practice, you'd extract these from a real Transformer model

tokens = ["The", "patient", "has", "a", "high", "fever", "and", "cough"]

# Simulated attention matrix (8x8) — one head of self-attention
np.random.seed(42)
raw_attention = np.random.rand(len(tokens), len(tokens))

# Make it more realistic: "fever" and "cough" attend to each other strongly
raw_attention[5, 7] = 3.0  # fever -> cough
raw_attention[7, 5] = 3.0  # cough -> fever
raw_attention[5, 4] = 2.5  # fever -> high
raw_attention[1, 5] = 2.0  # patient -> fever
raw_attention[1, 7] = 2.0  # patient -> cough

# Apply softmax per row
def softmax(x):
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e_x / e_x.sum(axis=-1, keepdims=True)

attention = softmax(raw_attention)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(attention, cmap='Blues', aspect='auto')
ax.set_xticks(range(len(tokens)))
ax.set_yticks(range(len(tokens)))
ax.set_xticklabels(tokens, rotation=45, ha='right')
ax.set_yticklabels(tokens)
ax.set_xlabel('Attended To (Keys)')
ax.set_ylabel('Attending From (Queries)')
ax.set_title('Self-Attention Weights (Simulated)')

# Add values
for i in range(len(tokens)):
    for j in range(len(tokens)):
        val = attention[i, j]
        color = 'white' if val > 0.2 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', color=color, fontsize=8)

plt.colorbar(im, label='Attention Weight')
plt.tight_layout()
plt.show()

print("Key observations:")
print("  - 'fever' strongly attends to 'cough' and 'high'")
print("  - 'patient' attends to symptom tokens")
print("  - In a real model, you'd visualize all heads and layers")

---
## 8. Model Card Template Generator

Generate a structured Model Card following Google's Model Card format.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


def generate_model_card(
    model_name,
    model_type,
    model,
    X_test,
    y_test,
    feature_names,
    intended_use,
    limitations,
    ethical_considerations,
    training_data_desc,
):
    """Generate a Model Card as a formatted string."""
    
    y_pred = model.predict(X_test)
    
    card = f"""
{'='*70}
MODEL CARD: {model_name}
{'='*70}

1. MODEL DETAILS
{'-'*40}
  Model Name:    {model_name}
  Model Type:    {model_type}
  Framework:     scikit-learn
  Version:       1.0
  Date Created:  2026-03-06
  License:       Internal Use Only

2. INTENDED USE
{'-'*40}
  Primary Use:   {intended_use}
  Users:         Data scientists, ML engineers, clinical researchers
  Out-of-scope:  Not intended for direct clinical decision-making
                 without human review.

3. TRAINING DATA
{'-'*40}
  Description:   {training_data_desc}
  Features:      {len(feature_names)} input features
  Top Features:  {', '.join(feature_names[:5])}

4. PERFORMANCE METRICS
{'-'*40}
  Accuracy:      {accuracy_score(y_test, y_pred):.4f}
  Precision:     {precision_score(y_test, y_pred):.4f}
  Recall:        {recall_score(y_test, y_pred):.4f}
  F1 Score:      {f1_score(y_test, y_pred):.4f}

5. LIMITATIONS
{'-'*40}
  {limitations}

6. ETHICAL CONSIDERATIONS
{'-'*40}
  {ethical_considerations}

7. INTERPRETABILITY
{'-'*40}
  Methods Used:  SHAP, LIME, Permutation Importance, PDP
  Explanation:   Feature attributions available for all predictions.
  Model Type:    {'Intrinsically interpretable' if model_type in ['LogisticRegression', 'DecisionTree'] else 'Post-hoc explanations required'}

{'='*70}
"""
    return card


# Generate Model Card for our Gradient Boosting model
card = generate_model_card(
    model_name="Breast Cancer Prediction Model",
    model_type="GradientBoostingClassifier",
    model=gb_model,
    X_test=X_test,
    y_test=y_test,
    feature_names=list(X.columns),
    intended_use="Binary classification of breast tumors as malignant or benign",
    limitations="Trained on UCI Breast Cancer dataset (569 samples). May not generalize\n"
               "  to populations not represented in the dataset. Performance may degrade\n"
               "  with different imaging equipment or measurement protocols.",
    ethical_considerations="Model predictions should supplement, not replace, clinical judgment.\n"
                          "  Ensure equitable performance across patient demographics.\n"
                          "  Regular monitoring for data drift is required.",
    training_data_desc="UCI Breast Cancer Wisconsin (Diagnostic) dataset, 569 samples, 30 features",
)

print(card)

In [ ]:
# Save Model Card to file
with open("model_card_breast_cancer.txt", "w") as f:
    f.write(card)

print("Model Card saved to 'model_card_breast_cancer.txt'")
print("\nIn production, you would attach this to Vertex AI Model Registry:")
print("  model.update(model_card=model_card_content)")

---
## Summary

| Technique | Scope | Speed | Any Model? | Key Strength |
|-----------|-------|-------|------------|---------------|
| Coefficients | Global | Instant | Linear only | Exact, built-in |
| Impurity Importance | Global | Instant | Trees only | Built-in, fast |
| SHAP | Global + Local | Slow (Kernel) / Fast (Tree) | Yes | Theoretically sound |
| LIME | Local | Fast | Yes | Intuitive |
| PDP | Global | Medium | Yes | Shows feature effects |
| Permutation | Global | Medium | Yes | Unbiased |
| Vertex AI Explain | Global + Local | Varies | Deployed models | Integrated with serving |

**Exam Tips:**
- SHAP is the gold standard for feature attribution (satisfies all axioms)
- Sampled Shapley works with ANY model on Vertex AI
- Integrated Gradients requires gradient access (TensorFlow)
- XRAI is for image models only
- Model Cards document model details, metrics, limitations, and ethics